# 4. Production Encoding with ColumnTransformer & Pipelines

This notebook demonstrates the end-to-end industry standard for categorical encoding:
1. Handling **multiple column types simultaneously** (Nominal, Ordinal, High-Cardinality, and Numerical).
2. Proper **Train/Test Splitting** to eliminate data leakage.
3. Fitting the transformation strictly on `X_train` and applying it to `X_test`.
4. Seamlessly chaining preprocessing with a Machine Learning model using `sklearn.pipeline.Pipeline`.

In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, TargetEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

# 1. Create a realistic enterprise dataset with mixed feature types
data = {
    'Age': [25, 34, 45, 22, 28, 52, 40, 29, 36, 48, 23, 41],
    'Annual_Income': [35000, 78000, 120000, 28000, 65000, 140000, 95000, 52000, 88000, 110000, 31000, 99000],
    'Device_Type': ['Android', 'iOS', 'Android', 'Windows', 'iOS', 'iOS', 'Android', 'Windows', 'iOS', 'Android', 'Android', 'iOS'], # Nominal
    'Education_Level': ['High School', 'Bachelors', 'PhD', 'High School', 'Masters', 'PhD', 'Bachelors', 'Masters', 'Bachelors', 'PhD', 'High School', 'Masters'], # Ordinal
    'City_Pincode': ['500001', '560001', '400001', '500001', '560001', '110001', '500001', '400001', '560001', '600001', '500001', '110001'], # High-Cardinality
    'Loan_Approved': [0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1] # Target (y)
}

df = pd.DataFrame(data)
print("=== 1. RAW MIXED DATASET ===")
display(df)
display(df.dtypes)

=== 1. RAW MIXED DATASET ===


,Age,Annual_Income,Device_Type,Education_Level,City_Pincode,Loan_Approved
0,25,35000,Android,High School,500001,0
1,34,78000,iOS,Bachelors,560001,1
2,45,120000,Android,PhD,400001,1
3,22,28000,Windows,High School,500001,0
4,28,65000,iOS,Masters,560001,1
5,52,140000,iOS,PhD,110001,1
6,40,95000,Android,Bachelors,500001,1
7,29,52000,Windows,Masters,400001,0
8,36,88000,iOS,Bachelors,560001,1
9,48,110000,Android,PhD,600001,1


Age                 int64
Annual_Income       int64
Device_Type        object
Education_Level    object
City_Pincode       object
Loan_Approved       int64
dtype: object

---
## Part 1: Train / Test Split (Preventing Data Leakage)

Before running any encoding or statistical transformations:
* We split data into training (`X_train`, `y_train`) and evaluation sets (`X_test`, `y_test`).
* Preprocessors must **never** see the test set during the `.fit()` stage.

In [6]:
X = df.drop(columns=['Loan_Approved']).copy()
y = df['Loan_Approved'].copy()

X_train , X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

print(f"Training shape: {X_train.shape} | Testing shape: {X_test.shape}")

Training shape: (9, 5) | Testing shape: (3, 5)
